In [29]:
!pip install pytesseract
!pip install tesseract

     ---------------------------------------- 45.6/45.6 MB 8.9 MB/s  0:00:05
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Created wheel for tesseract: filename=tesseract-0.1.3-py3-none-any.whl size=45562602 sha256=2fe5f66e6ee1264c957048813e73d94c5c5ecdfc813c868ca1339dd644913791
  Stored in directory: c:\users\gidle\appdata\local\pip\cache\wheels\6c\c5\81\8310cc52076953e53412ed1875a5e224c92940235bdcee21a2
Successfully built tesseract


In [30]:
import os
import cv2
import numpy as np
import pytesseract
import matplotlib.pyplot as plt
from random import sample

In [31]:
target_dir_inpainted = '../data/hateful_memes_inpainted'
source_dir = 'C:/Users/gidle/Desktop/Year3/NLP/Project/Hateful memes challenge/DATA/img'
target_dir_masked = 'C:/Users/gidle/Desktop/Year3/NLP/Project/Hateful memes challenge/DATA/masked'
target_dir_inpainted = 'C:/Users/gidle/Desktop/Year3/NLP/Project/Hateful memes challenge/DATA/inpainted'
img_fns = [x for x in os.listdir(source_dir) if x.lower().endswith(('.png', '.jpg', '.jpeg'))]

In [32]:
east_path = 'C:/Users/gidle/Desktop/Year3/NLP/Project/Hateful memes challenge/MODEL/frozen_east_text_detection.pb'
width = 640
height = 640
min_confidence = 0.4

In [33]:
layerNames = [
    "feature_fusion/Conv_7/Sigmoid",
    "feature_fusion/concat_3"
]

print("[INFO] loading EAST text detector...")
net = cv2.dnn.readNet(east_path)

[INFO] loading EAST text detector...


In [34]:
def is_added_text(text_region, original_image):
    """
    判断文字是添加的文字还是背景元素
    基于以下特征：
    1. 文字区域的颜色一致性
    2. 文字边缘的锐利程度
    3. 与周围背景的对比度
    4. OCR识别置信度
    """
    x1, y1, x2, y2 = text_region
    text_patch = original_image[y1:y2, x1:x2]
    
    # 1. 颜色标准差（添加的文字通常颜色单一）
    color_std = np.std(text_patch, axis=(0,1))
    avg_color_std = np.mean(color_std)
    
    # 2. 边缘梯度（添加的文字通常边缘更锐利）
    gray = cv2.cvtColor(text_patch, cv2.COLOR_BGR2GRAY)
    grad_x = cv2.Sobel(gray, cv2.CV_64F, 1, 0, ksize=3)
    grad_y = cv2.Sobel(gray, cv2.CV_64F, 0, 1, ksize=3)
    edge_strength = np.mean(np.abs(grad_x)) + np.mean(np.abs(grad_y))
    
    # 3. 与周围背景的对比度
    expanded_region = original_image[max(0,y1-5):min(original_image.shape[0],y2+5),
                                    max(0,x1-5):min(original_image.shape[1],x2+5)]
    text_mean = np.mean(gray)
    bg_mean = np.mean(cv2.cvtColor(expanded_region, cv2.COLOR_BGR2GRAY))
    contrast = abs(text_mean - bg_mean)
    
    # 4. OCR识别（添加的文字通常更清晰可读）
    text = pytesseract.image_to_string(text_patch, config='--psm 7')
    ocr_confidence = len(text.strip()) > 0  # 简单判断是否有识别结果
    
    # 综合判断（可根据实际情况调整阈值）
    is_added = (avg_color_std < 30 and edge_strength > 50 and 
                contrast > 60 and ocr_confidence)
    
    return is_added


In [35]:
def transform_image(img_fp):
    # 加载原始图像
    original_image = cv2.imread(img_fp)
    if original_image is None:
        raise ValueError(f"无法加载图像: {img_fp}")
    
    (H_orig, W_orig) = original_image.shape[:2]
    
    # 调整图像大小以适应EAST模型
    image = cv2.resize(original_image, (width, height))
    (H, W) = image.shape[:2]
    
    # 计算原始图像与调整后图像的比例
    rW = W_orig / float(W)
    rH = H_orig / float(H)
    
    # 构造blob并前向传播
    blob = cv2.dnn.blobFromImage(image, 1.0, (W, H),
                                (123.68, 116.78, 103.94), swapRB=True, crop=False)
    net.setInput(blob)
    (scores, geometry) = net.forward(layerNames)
    
    # 解析检测结果
    (numRows, numCols) = scores.shape[2:4]
    rects = []
    confidences = []
    
    for y in range(0, numRows):
        scoresData = scores[0, 0, y]
        xData0 = geometry[0, 0, y]
        xData1 = geometry[0, 1, y]
        xData2 = geometry[0, 2, y]
        xData3 = geometry[0, 3, y]
        anglesData = geometry[0, 4, y]

        for x in range(0, numCols):
            if scoresData[x] < min_confidence:
                continue
                
            (offsetX, offsetY) = (x * 4.0, y * 4.0)
            angle = anglesData[x]
            cos = np.cos(angle)
            sin = np.sin(angle)
            
            h = xData0[x] + xData2[x]
            w = xData1[x] + xData3[x]
            
            endX = int(offsetX + (cos * xData1[x]) + (sin * xData2[x]))
            endY = int(offsetY - (sin * xData1[x]) + (cos * xData2[x]))
            startX = int(endX - w)
            startY = int(endY - h)
            
            rects.append((startX, startY, endX, endY))
            confidences.append(scoresData[x])
    
    # 合并同一行的文本框
    merged_boxes = []
    if len(rects) > 0:
        rects = np.array(rects)
        
        # 计算每行文字的平均高度
        centers_y = (rects[:, 1] + rects[:, 3]) / 2
        heights = rects[:, 3] - rects[:, 1]
        avg_height = np.mean(heights)
        
        # 根据y坐标分组
        rows = []
        current_row = [rects[0]]
        for i in range(1, len(rects)):
            if abs(centers_y[i] - centers_y[i-1]) < avg_height / 2:
                current_row.append(rects[i])
            else:
                rows.append(current_row)
                current_row = [rects[i]]
        rows.append(current_row)
        
        # 合并每行的文本框
        for row in rows:
            row = np.array(row)
            min_x = np.min(row[:, 0])
            min_y = np.min(row[:, 1])
            max_x = np.max(row[:, 2])
            max_y = np.max(row[:, 3])
            merged_boxes.append([min_x, min_y, max_x, max_y])

In [25]:

layerNames = [
	"feature_fusion/Conv_7/Sigmoid",
	"feature_fusion/concat_3"
    ]
print("[INFO] loading EAST text detector...")
net = cv2.dnn.readNet(east_path)


[INFO] loading EAST text detector...


In [36]:
def transform_image(img_fp):
    # 加载原始图像
    original_image = cv2.imread(img_fp)
    if original_image is None:
        raise ValueError(f"无法加载图像: {img_fp}")
    
    (H_orig, W_orig) = original_image.shape[:2]
    
    # 调整图像大小以适应EAST模型
    image = cv2.resize(original_image, (width, height))
    (H, W) = image.shape[:2]
    
    # 计算原始图像与调整后图像的比例
    rW = W_orig / float(W)
    rH = H_orig / float(H)
    
    # 构造blob并前向传播
    blob = cv2.dnn.blobFromImage(image, 1.0, (W, H),
                                (123.68, 116.78, 103.94), swapRB=True, crop=False)
    net.setInput(blob)
    (scores, geometry) = net.forward(layerNames)
    
    # 解析检测结果
    (numRows, numCols) = scores.shape[2:4]
    rects = []
    confidences = []
    
    for y in range(0, numRows):
        scoresData = scores[0, 0, y]
        xData0 = geometry[0, 0, y]
        xData1 = geometry[0, 1, y]
        xData2 = geometry[0, 2, y]
        xData3 = geometry[0, 3, y]
        anglesData = geometry[0, 4, y]

        for x in range(0, numCols):
            if scoresData[x] < min_confidence:
                continue
                
            (offsetX, offsetY) = (x * 4.0, y * 4.0)
            angle = anglesData[x]
            cos = np.cos(angle)
            sin = np.sin(angle)
            
            h = xData0[x] + xData2[x]
            w = xData1[x] + xData3[x]
            
            endX = int(offsetX + (cos * xData1[x]) + (sin * xData2[x]))
            endY = int(offsetY - (sin * xData1[x]) + (cos * xData2[x]))
            startX = int(endX - w)
            startY = int(endY - h)
            
            rects.append((startX, startY, endX, endY))
            confidences.append(scoresData[x])
    
    # 合并同一行的文本框
    merged_boxes = []
    if len(rects) > 0:
        rects = np.array(rects)
        
        # 计算每行文字的平均高度
        centers_y = (rects[:, 1] + rects[:, 3]) / 2
        heights = rects[:, 3] - rects[:, 1]
        avg_height = np.mean(heights)
        
        # 根据y坐标分组
        rows = []
        current_row = [rects[0]]
        for i in range(1, len(rects)):
            if abs(centers_y[i] - centers_y[i-1]) < avg_height / 2:
                current_row.append(rects[i])
            else:
                rows.append(current_row)
                current_row = [rects[i]]
        rows.append(current_row)
        
        # 合并每行的文本框
        for row in rows:
            row = np.array(row)
            min_x = np.min(row[:, 0])
            min_y = np.min(row[:, 1])
            max_x = np.max(row[:, 2])
            max_y = np.max(row[:, 3])
            merged_boxes.append([min_x, min_y, max_x, max_y])
            # 准备输出图像
    masked = original_image.copy()
    inpainted = original_image.copy()
    classified_img = original_image.copy()
    mask_for_inpainting = np.zeros(inpainted.shape[:2], np.uint8)
    
    # 分类文本框并处理
    classified_boxes = {'added': [], 'background': []}
    for box in merged_boxes:
        startX, startY, endX, endY = box
        
        # 缩放回原始图像尺寸
        orig_startX = int(startX * rW)
        orig_startY = int(startY * rH)
        orig_endX = int(endX * rW)
        orig_endY = int(endY * rH)
        
        # 分类文字类型
        if is_added_text((orig_startX, orig_startY, orig_endX, orig_endY), original_image):
            classified_boxes['added'].append((orig_startX, orig_startY, orig_endX, orig_endY))
            # 对添加的文字进行掩码和修复
            cv2.rectangle(masked, (orig_startX, orig_startY), (orig_endX, orig_endY), (127, 127, 127), -1)
            cv2.rectangle(mask_for_inpainting, (orig_startX, orig_startY), (orig_endX, orig_endY), 255, -1)
        else:
            classified_boxes['background'].append((orig_startX, orig_startY, orig_endX, orig_endY))
    
    # 应用图像修复
    inpainted = cv2.inpaint(inpainted, mask_for_inpainting, 7, cv2.INPAINT_NS)
    
    # 在分类图像上绘制不同颜色的框
    for box in classified_boxes['added']:
        cv2.rectangle(classified_img, (box[0], box[1]), (box[2], box[3]), (0, 255, 0), 2)  # 绿色-添加文字
    for box in classified_boxes['background']:
        cv2.rectangle(classified_img, (box[0], box[1]), (box[2], box[3]), (0, 0, 255), 2)  # 红色-背景文字
    
    return masked, inpainted, classified_img, classified_boxes

In [37]:
def visualize_text_detection_results(original_img_path):
    """
    可视化文本检测和分类结果
    """
    masked, inpainted, classified_img, classified_boxes = transform_image(original_img_path)
    original = cv2.imread(original_img_path)
    
    # 转换颜色空间用于显示
    original_rgb = cv2.cvtColor(original, cv2.COLOR_BGR2RGB)
    masked_rgb = cv2.cvtColor(masked, cv2.COLOR_BGR2RGB)
    inpainted_rgb = cv2.cvtColor(inpainted, cv2.COLOR_BGR2RGB)
    classified_rgb = cv2.cvtColor(classified_img, cv2.COLOR_BGR2RGB)
    
    # 创建可视化图表
    plt.figure(figsize=(24, 6))
    
    plt.subplot(1, 4, 1)
    plt.imshow(original_rgb)
    plt.title('Original Image')
    plt.axis('off')
    
    plt.subplot(1, 4, 2)
    plt.imshow(masked_rgb)
    plt.title('Masked Added Text')
    plt.axis('off')

    plt.subplot(1, 4, 3)
    plt.imshow(inpainted_rgb)
    plt.title('Inpainted Image')
    plt.axis('off')
    
    plt.subplot(1, 4, 4)
    plt.imshow(classified_rgb)
    plt.title('Text Classification\n(Green=Added, Red=Background)')
    plt.axis('off')
    
    plt.tight_layout()
    plt.show()
    
    # 打印分类统计信息
    print(f"Detected {len(classified_boxes['added'])} added text regions")
    print(f"Detected {len(classified_boxes['background'])} background text elements")


# 随机选择2张图像进行测试
sample_images = sample(img_fns, 2) if len(img_fns) >= 2 else img_fns

for img_fn in sample_images:
    img_path = os.path.join(source_dir, img_fn)
    print(f"\nProcessing image: {img_fn}")
    try:
        visualize_text_detection_results(img_path)
    except Exception as e:
        print(f"Error processing {img_fn}: {str(e)}")


Processing image: 30697.png
Error processing 30697.png: tesseract is not installed or it's not in your PATH. See README file for more information.

Processing image: 24501.png
Error processing 24501.png: tesseract is not installed or it's not in your PATH. See README file for more information.
